In [1]:
import torch 
import numpy as np 
import h5py
import os
import IPython.display as ipd
from corpus.speaker_room_dataset import SpeakerRoomDataset
import src.audio_transforms as at

import src.spatial_attn_lightning as binaural_lightning 
import yaml
from tqdm.auto import tqdm

%matplotlib inline
import matplotlib.pyplot as plt

os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

torch.set_float32_matmul_precision('medium')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
!hostname

node078


### Approach for time shift - grid search across onset times

In [3]:
### Get most recent config
# config_path = "config/binaural_attn/word_task_25p_loc_v07_LN_last_valid_time_no_affine.yaml"
# ckpt_path = "attn_cue_models/word_task_25p_loc_v07_LN_last_valid_time_no_affine/checkpoints/epoch=3-step=49432.ckpt"
# old_style = False
### Get most recent config
config_path = "config/binaural_attn/word_task_half_co_loc_v07.yaml"
ckpt_path = "attn_cue_models/word_task_half_co_loc_v07/checkpoints/epoch=2-step=46074.ckpt"
# old_style = True 

config = yaml.load(open(config_path, 'r'), Loader=yaml.FullLoader)

config['hparas']['batch_size'] = 2 # config['data']['loader']['batch_size'] // args.gpus
config['num_workers'] = 2
config['noise_kwargs']['low_snr'] = 0
config['noise_kwargs']['high_snr'] = 0
# get model input sr for brir resampling
signal_sr = config['audio']['rep_kwargs']['sr']
coch_sr = config['audio']['rep_kwargs']['env_sr']
# cue_duration = 0.5
# n_cue_frames = int(cue_duration * signal_sr)
# config['model']['n_cue_frames'] = n_cue_frames

# config['cue_duration_test'] = True


In [4]:
model = binaural_lightning.BinauralAttentionModule.load_from_checkpoint(checkpoint_path=ckpt_path, config=config).cuda().eval()


num_classes={'num_words': 800}


/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/site-packages/torchaudio/functional/functional.py:1371: UserWarning: "kaiser_window" resampling method name is being deprecated and replaced by "sinc_interp_kaiser" in the next release. The default behavior remains unchanged.
  warnings.warn(


center_crop=True
binaural=True
Binaural cochleagram
using FIR cochleagram


In [5]:
dataset = SpeakerRoomDataset(manifest_path='/om2/user/rphess/Auditory-Attention/final_binaural_manifest.pkl',
                            excerpt_path='/om2/user/msaddler/spatial_audio_pipeline/assets/swc/manifest_all_words.pdpkl',
                            cue_type='voice_and_location',
                            sr=signal_sr) 


diotic_transforms = at.AudioCompose([
                    at.AudioToTensor(),
                    at.CombineWithRandomDBSNR(low_snr=0, high_snr=0), 
                    at.RMSNormalizeForegroundAndBackground(rms_level=0.02),  # 0.02 is the default for CV-based models 
                    at.DuplicateChannel(),
            ])
copy_channel_and_norm = at.AudioCompose([
                    at.AudioToTensor(),
                    at.DuplicateChannel(),
                    at.BinauralRMSNormalizeForegroundAndBackground(rms_level=0.02, v2_demean=True)
            ])
rms_norm =  at.BinauralRMSNormalizeForegroundAndBackground(rms_level=0.02, v2_demean=True)

diotic_transforms = diotic_transforms.cuda()


## Define alternate audio transforms to see effect on generated distractors

In [6]:
audio_transforms = at.AudioCompose([
                at.AudioToTensor(),
                at.BinauralCombineWithRandomDBSNR(low_snr=0,
                                                  high_snr=0,
                                                  v2_demean=True),
                at.BinauralRMSNormalizeForegroundAndBackground(rms_level=0.02, v2_demean=True), # 20 * np.log10(0.02/20e-6) = 60 dB SPL 
            ])


In [7]:
import pickle
class_map = pickle.load( open("/om2/user/imgriff/datasets/commonvoice_9/en/cv_800_word_label_to_int_dict.pkl", "rb" )) 
ix_to_word = {v: k for k, v in class_map.items()}

In [11]:
import numpy as np
import matplotlib.pyplot as plt

# Set up parameters
RANDOMSEED = 0
SAMPLERATE = 44_100
signal_len = 3 # in seconds
noise_len = 3 # in seconds 
noise_eps = 5e-2 # noise floor used in distractor signal

np.random.seed(RANDOMSEED)

# Initialize distractor signal
distractor = np.random.randn(int(SAMPLERATE * noise_len))
Q = 20
cos_window = np.hanning(int(SAMPLERATE * noise_len))
distractor = distractor * cos_window**Q
distractor = distractor + torch.rand_like(distractor) * noise_eps

# Manually shift the middle point of the distractor signal back by one second 
distractor = np.roll(distractor, int(SAMPLERATE * -1))

ix = 10
cue, fg, bg, label, confusion = dataset[ix]

cue, _ = diotic_transforms(cue, None)
fg, _ = diotic_transforms(fg, None)

cue = cue.unsqueeze(0)
fg = fg.unsqueeze(0)
bg = bg.unsqueeze(0)
label = torch.tensor(label).unsqueeze(0)

loss_fn = torch.nn.CrossEntropyLoss()

coch_transform = model.coch_gram.cuda()


n_steps = 1000
best_loss = 1.0
best_distractor = None
best_step = 0 

# Grid search over possible distractor signal onsets
for onset in np.arange(0, distractor.shape[0], SAMPLERATE):
    distractor_shifted = np.roll(distractor, onset)
    distractor_eg_fin, _ = copy_channel_and_norm(distractor_shifted, None)

    mixture_wav, _ = audio_transforms(fg, distractor_eg_fin)
    mixture, _ = coch_transform(mixture_wav, None)

    logits = model(cue, mixture, None)
    loss = -loss_fn(logits, label)

    if loss < best_loss:
        best_loss = loss.detach().item()
        best_distractor = distractor_shifted
        best_step = onset

print(f"Best loss: {best_loss:.4f} at onset: {best_step}")

TypeError: rand_like(): argument 'input' (position 1) must be Tensor, not numpy.ndarray